# 04 · Git and GitHub

**Sessions 4 and 6 — Wrangle and Version · Branch and Plot** · about 30 minutes

**You will learn to**
- explain Git's four areas: working directory, staging area, local repository, remote
- stage, commit, inspect history and differences
- work on branches, merge them and resolve a conflict
- connect a repository to GitHub with a personal access token

**Data:** none: everything happens in a temporary sandbox folder, never in the course repository.

**Outline**
0. Before you start (no AI)
1. A sandbox repository
2. Stage and commit
3. Branches and merges
4. A merge conflict
5. GitHub
6. Clean up

> **AI guardrails** — *Think first · Own it · Verify · Disclose · Protect.* AI tools are allowed in work time, but answer the **No-AI** questions on paper first, and make every ✅ self-check pass with code you can explain line by line.

## 0. Before you start (no AI)

Draw four boxes — **working directory**, **staging area**, **local repository**, **remote (GitHub)** —
and arrows for `git add`, `git commit`, `git push`, `git pull`.

Then answer: *you committed, but your teammate does not see your change on GitHub. Why?*

## 1. A sandbox repository

We run Git through a small Python helper so it works the same on Windows, macOS and Linux,
and only ever touches a temporary folder. (In a terminal you would type the same commands
without `git(...)`.)

> You need Git installed (`git --version`). On Windows: *Git for Windows*.

In [ ]:
import shutil
import subprocess
import tempfile
from pathlib import Path

SANDBOX = Path(tempfile.mkdtemp(prefix="git-sandbox-"))
IDENTITY = ["-c", "user.name=Course Student", "-c", "user.email=student@example.com"]


def git(*args, check=True):
    """Run a git command inside SANDBOX and print what Git prints."""
    result = subprocess.run(["git", *IDENTITY, *args], cwd=SANDBOX, capture_output=True, text=True)
    print(f"$ git {' '.join(args)}")
    print((result.stdout + result.stderr).strip() or "(no output)")
    if check and result.returncode != 0:
        raise RuntimeError(f"git {' '.join(args)} failed")
    return result


def write(name, text):
    (SANDBOX / name).write_text(text, encoding="utf-8")


print("Sandbox:", SANDBOX)
git("init", "-b", "main")

`IDENTITY` sets a name and e-mail only for these commands. On your own computer, set yours once:

```console
git config --global user.name "Your Name"
git config --global user.email "you@example.com"
```

## 2. Stage and commit

*Predict first:* what will `git status` say right after creating `README.md`?

In [ ]:
write("README.md", "# Air quality in Barcelona\n")
git("status", "--short")

`??` = untracked: Git sees the file but is not tracking it. Stage it, then commit it:

In [ ]:
git("add", "README.md")
git("status", "--short")          # 'A' = added to the staging area
git("commit", "-m", "Add README with the project title")
git("log", "--oneline")

The commit now lives in the **local repository** (the hidden `.git` folder). Nobody else can see it
until you `git push`.

Change the file and look at the difference before committing:

In [ ]:
write("README.md", "# Air quality in Barcelona\n\nData: Open Data BCN.\n")
git("diff")
git("commit", "-am", "Document the data source")
git("log", "--oneline")

## 3. Branches and merges

Never work directly on `main`: create a branch for each piece of work, then merge it.
`git switch -c` is the modern form of `git checkout -b`.

In [ ]:
git("switch", "-c", "feature/first-plot")
write("plot.py", "import matplotlib.pyplot as plt\n")
git("add", "plot.py")
git("commit", "-m", "Start the first plot script")
git("switch", "main")
print("plot.py on main?", (SANDBOX / "plot.py").exists())
git("merge", "feature/first-plot")
print("plot.py on main after the merge?", (SANDBOX / "plot.py").exists())

## 4. A merge conflict

Two branches change **the same line**. Git cannot choose for you.

In [ ]:
git("switch", "-c", "feature/title-a")
write("README.md", "# Air quality in Barcelona, 2019–2024\n\nData: Open Data BCN.\n")
git("commit", "-am", "Add the period to the title")

git("switch", "main")
git("switch", "-c", "feature/title-b")
write("README.md", "# NO2 levels in Barcelona\n\nData: Open Data BCN.\n")
git("commit", "-am", "Name the pollutant in the title")

git("switch", "main")
git("merge", "feature/title-a")
git("merge", "feature/title-b", check=False)       # conflict!
print((SANDBOX / "README.md").read_text())

Git wrote both versions between `<<<<<<<`, `=======` and `>>>>>>>`.
**Resolve** by editing the file to the version you want, then `add` and `commit`.

### TASK — resolve the conflict

Write the final title you want into `README.md` (keep the *Data* line), stage it and commit
with the message `"Merge title changes"`. Use the `write` and `git` helpers.

In [ ]:
# TASK — your code here

In [ ]:
# ✅ Self-check
text = (SANDBOX / "README.md").read_text()
assert "<<<<<<<" not in text and ">>>>>>>" not in text, "Remove the conflict markers."
assert "Data: Open Data BCN." in text, "Keep the data line."
status = subprocess.run(["git", "status", "--porcelain"], cwd=SANDBOX, capture_output=True, text=True).stdout
assert status.strip() == "", "Commit the resolution."
print("✅ Conflict resolved")

## 5. GitHub

GitHub hosts the **remote**. Over HTTPS it asks for a **personal access token**, not your password
([how to create one](https://docs.github.com/en/authentication/keeping-your-account-and-data-secure/managing-your-personal-access-tokens)).

The workflow for your group repository (in a terminal, inside your project folder):

```console
git clone https://github.com/<org>/<repo>.git     # first time only
git pull                                         # get your teammates' work first
git switch -c feature/<what-you-do>
git add <files> && git commit -m "Say why"
git push -u origin feature/<what-you-do>          # then open a Pull Request on GitHub
```

On GitHub: *Compare & pull request* → ask a teammate to review → *Merge* → everyone `git pull`s `main`.

Practise branching interactively at [Learn Git Branching](https://learngitbranching.js.org/).

## 6. Clean up

In [ ]:
shutil.rmtree(SANDBOX, ignore_errors=True)
print("Sandbox removed:", not SANDBOX.exists())

**Next:** `05_matplotlib_canvas.ipynb` (Session 6).